# CESLR Training on Google Colab

Train **Double Cosign** on pre-extracted Pose86 keypoints — no raw video frames or pose extraction needed.

**Before running:** enable a GPU runtime (`Runtime → Change runtime type → T4 GPU`).

**Repo:** [Skeleton-Based-Continuous-Ethiopia-Sign-Language-Recognition](https://github.com/ethio-artifical/Skeleton-Based-Continuous-Ethiopia-Sign-Language-Recognition)

Data included in the clone:
- `datasets/pose_data_ceslr_hands_lips_body.pkl` (~20 MB, 214 videos)
- `datasets/ceslr/` — train/dev/test splits, gloss dict, STM ground truth

## 1. GPU check

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone repository

In [ ]:
import os

REPO = "Skeleton-Based-Continuous-Ethiopia-Sign-Language-Recognition"
if not os.path.isdir(REPO):
    !git clone https://github.com/ethio-artifical/Skeleton-Based-Continuous-Ethiopia-Sign-Language-Recognition.git

%cd {REPO}

## 3. Verify keypoints and annotations

In [ ]:
import os
import pickle

POSE_PKL = "datasets/pose_data_ceslr_hands_lips_body.pkl"
GLOSS_JSON = "datasets/ceslr/gloss_dict.json"

assert os.path.exists(POSE_PKL), f"Missing {POSE_PKL} — push it from your local machine first"
assert os.path.exists(GLOSS_JSON), f"Missing {GLOSS_JSON}"

with open(POSE_PKL, "rb") as f:
    pose_data = pickle.load(f)

print(f"Videos in pose pkl: {len(pose_data)} (expect 214)")
print(f"Gloss dict: {GLOSS_JSON}")
for split in ("train", "dev", "test"):
    path = f"datasets/ceslr/{split}_info.json"
    assert os.path.exists(path), f"Missing {path}"
    print(f"  {split}_info.json OK")

## 4. Install training dependencies

MediaPipe and OpenCV are **not** required — keypoints are already in the repo.

`ctcdecode` is optional; training falls back to greedy decode if the build fails.

In [ ]:
!pip install -q torch torchvision pyyaml tqdm matplotlib

# Optional beam search (greedy fallback if this fails)
!apt-get -y -qq install build-essential
!pip install -q ctcdecode --no-build-isolation || echo "ctcdecode install failed — using greedy decode"

## 5. Train

Config: `configs/Double_Cosign_ceslr.yaml` — 40 epochs, batch size 4.

Outputs:
- Logs: `work_dir/ceslr/log.txt`
- Checkpoints: `work_dir/ceslr/best_dev_*.pt`

If you hit GPU OOM, add `--batch-size 2`.

In [ ]:
!python main.py --config ./configs/Double_Cosign_ceslr.yaml --num-worker 2

## 6. (Optional) Save checkpoints to Google Drive

Run this before the Colab session times out to keep your trained weights.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import shutil
from pathlib import Path

src = Path("work_dir/ceslr")
dst = Path("/content/drive/MyDrive/ceslr_training_output")

if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Copied {src} → {dst}")
else:
    print(f"No outputs at {src} — run training first")